# OpenDataZurich starter notebook

This notebook runs **entirely in your browser** (JupyterLite with a Pyodide Python kernel): pandas, matplotlib, seaborn and the OpenDataZurich helper are preloaded when the kernel starts, and your changes are stored only in this browser. Run the cells from top to bottom with <kbd>Shift</kbd>+<kbd>Enter</kbd>, or use *Run → Run All Cells*.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from opendata_lite import OpenDataZurich

odz = OpenDataZurich()

In [ ]:
package_id = "bau_hae_lima_zuordnung_adr_quartier_bzo16_bzo99_od5143"
resource_id = None

In [ ]:
# Use the name of a dataset, e.g., from the OpenDataZurich site or the list above, to get the package
package = await odz.get_package(name=package_id)

In [ ]:
package.display_metadata()

In [ ]:
package.display_resource_summary()

In [ ]:
package.tabular_resource_metadata_df.head()[["name", "format"]]

In [ ]:
resource = package.tabular_resource(id=resource_id) if resource_id is not None and resource_id.upper() != "NONE" else package.tabular_resource(0)
resource.display_metadata()

In [ ]:
df = await resource.load()

In [ ]:
# drop columns that have no values
df.dropna(how='all', axis=1, inplace=True)

In [ ]:
print(f'The dataset has {df.shape[0]:,.0f} rows (observations) and {df.shape[1]:,.0f} columns (variables).')
print(f'There seem to be {df.duplicated().sum()} exact duplicates in the data.')

In [ ]:
df.info(memory_usage='deep', verbose=True)

In [ ]:
df.head()

In [ ]:
# display a small random sample transposed in order to see all variables
df.sample(3).T

In [ ]:
# describe non-numerical features
try:
    with pd.option_context('display.float_format', '{:,.2f}'.format):
        display(df.describe(exclude='number'))
except:
    print("No categorical data in dataset.")

In [ ]:
# describe numerical features
try:
    with pd.option_context('display.float_format', '{:,.2f}'.format):
        display(df.describe(include='number'))
except:
    print("No numercial data in dataset.")

In [ ]:
# Column types, completeness, and distinct non-missing values.
profile = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=True),
    }
).sort_values("missing_pct", ascending=False)
display(profile.round(1))

In [ ]:
# Show up to 20 columns with missing values; percentages use all rows.
missing = profile.loc[profile["missing"] > 0].head(20).rename_axis("column").reset_index()
if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, max(2, len(missing) * 0.3)))
    ax.barh(missing["column"], missing["missing_pct"], color="steelblue")
    ax.set(xlim=(0, 100), xlabel="Missing values (%)", ylabel="", title="Missingness by column")
    fig.tight_layout()
    plt.show()
else:
    print("No missing values to plot (or the dataset has no rows).")

In [ ]:
# plot a histogram for each numerical feature
try:
    df.hist(bins=25, rwidth=.9)
    plt.tight_layout()
    plt.show()
except:
    print("No numercial data to plot.")

## Need more power, or R?

This in-browser kernel is limited to the memory of your browser tab (a few GB at most) and cannot install packages with compiled code that are not part of Pyodide. For large datasets, other Python packages or for working in R, open the same dataset in a full Renku session:

* Python: <https://renkulab.io/p/opendatazurich/starter-code>
* R: <https://github.com/opendatazurich/opendatazurich_renku_r>

Use *File → Download* to keep a copy of this notebook.